In [ ]:
"""
Marathi Speech-to-Text Evaluation using AI4Bharat IndicConformer
This script evaluates 15 Marathi audio samples using AI4Bharat's IndicConformer STT model
"""

import torch
import torchaudio
from transformers import AutoModel
import os
from typing import List, Dict
import json

class MarathiSTTEvaluator:
    """
    Evaluator for Marathi Speech-to-Text using AI4Bharat IndicConformer
    Model: ai4bharat/indic-conformer-600m-multilingual
    """

    def __init__(self, model_name: str = "ai4bharat/indic-conformer-600m-multilingual"):
        """
        Initialize the Marathi STT model

        Args:
            model_name: HuggingFace model identifier
        """
        print(f"Loading model: {model_name}")
        self.model = AutoModel.from_pretrained(model_name, trust_remote_code=True , token = "")
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = self.model.to(self.device)
        print(f"Model loaded on device: {self.device}")

    def preprocess_audio(self, audio_path: str):
        """
        Preprocess audio file for model input

        Args:
            audio_path: Path to audio file

        Returns:
            Preprocessed audio tensor
        """
        # Load audio file
        wav, sr = torchaudio.load(audio_path)

        # Convert stereo to mono if needed
        if wav.shape[0] > 1:
            wav = torch.mean(wav, dim=0, keepdim=True)

        # Resample to 16kHz if needed
        target_sample_rate = 16000
        if sr != target_sample_rate:
            resampler = torchaudio.transforms.Resample(
                orig_freq=sr,
                new_freq=target_sample_rate
            )
            wav = resampler(wav)

        return wav, target_sample_rate

    def transcribe(self, audio_path: str, language: str = "mr") -> str:
        """
        Transcribe audio file to text

        Args:
            audio_path: Path to audio file
            language: Language code (mr for Marathi)

        Returns:
            Transcribed text
        """
        # Preprocess audio
        wav, sr = self.preprocess_audio(audio_path)

        # Transcribe using CTC decoder
        with torch.no_grad():
            transcript = self.model(
                wav,
                sr,
                "mr",
                "ctc"
            )

        return transcript

    def evaluate_batch(self, audio_files: List[str],
                      ground_truths: List[str] = None) -> List[Dict]:
        """
        Evaluate multiple audio files

        Args:
            audio_files: List of audio file paths
            ground_truths: Optional list of ground truth transcriptions

        Returns:
            List of evaluation results
        """
        results = []

        for idx, audio_file in enumerate(audio_files):
            print(f"\nProcessing audio {idx + 1}/{len(audio_files)}: {audio_file}")

            try:
                # Transcribe audio
                transcript = self.transcribe(audio_file)

                result = {
                    "audio_file": audio_file,
                    "transcript": transcript,
                    "status": "success"
                }

                # Add ground truth if available
                if ground_truths and idx < len(ground_truths):
                    result["ground_truth"] = ground_truths[idx]

                    # Calculate simple word error rate (WER)
                    result["wer"] = self.calculate_wer(
                        ground_truths[idx],
                        transcript
                    )

                results.append(result)
                print(f"Transcript: {transcript}")

            except Exception as e:
                print(f"Error processing {audio_file}: {str(e)}")
                results.append({
                    "audio_file": audio_file,
                    "transcript": "",
                    "status": "error",
                    "error": str(e)
                })

        return results

    def calculate_wer(self, reference: str, hypothesis: str) -> float:
        """
        Calculate Word Error Rate

        Args:
            reference: Ground truth text
            hypothesis: Predicted text

        Returns:
            WER score
        """
        ref_words = reference.split()
        hyp_words = hypothesis.split()

        # Simple edit distance calculation
        d = [[0] * (len(hyp_words) + 1) for _ in range(len(ref_words) + 1)]

        for i in range(len(ref_words) + 1):
            d[i][0] = i
        for j in range(len(hyp_words) + 1):
            d[0][j] = j

        for i in range(1, len(ref_words) + 1):
            for j in range(1, len(hyp_words) + 1):
                if ref_words[i-1] == hyp_words[j-1]:
                    d[i][j] = d[i-1][j-1]
                else:
                    d[i][j] = min(d[i-1][j], d[i][j-1], d[i-1][j-1]) + 1

        wer = d[len(ref_words)][len(hyp_words)] / len(ref_words) if ref_words else 0
        return wer

    def save_results(self, results: List[Dict], output_file: str = "evaluation_results.txt"):
        """
        Save evaluation results to text file

        Args:
            results: Evaluation results
            output_file: Output file path
        """
        with open(output_file, 'w', encoding='utf-8') as f:
            f.write("=" * 80 + "\n")
            f.write("Marathi Speech-to-Text Evaluation Results\n")
            f.write("Model: AI4Bharat IndicConformer (600M Multilingual)\n")
            f.write("=" * 80 + "\n\n")

            for idx, result in enumerate(results, 1):
                f.write(f"Audio {idx}: {result['audio_file']}\n")
                f.write(f"Status: {result['status']}\n")

                if result['status'] == 'success':
                    f.write(f"Transcript: {result['transcript']}\n")

                    if 'ground_truth' in result:
                        f.write(f"Ground Truth: {result['ground_truth']}\n")
                        f.write(f"WER: {result['wer']:.4f}\n")
                else:
                    f.write(f"Error: {result.get('error', 'Unknown error')}\n")

                f.write("-" * 80 + "\n\n")

            # Summary statistics
            successful = sum(1 for r in results if r['status'] == 'success')
            f.write("\n" + "=" * 80 + "\n")
            f.write("Summary\n")
            f.write("=" * 80 + "\n")
            f.write(f"Total audio files: {len(results)}\n")
            f.write(f"Successfully processed: {successful}\n")
            f.write(f"Failed: {len(results) - successful}\n")

            if any('wer' in r for r in results):
                avg_wer = sum(r.get('wer', 0) for r in results if 'wer' in r) / sum(1 for r in results if 'wer' in r)
                f.write(f"Average WER: {avg_wer:.4f}\n")

        print(f"\nResults saved to {output_file}")


def main():
    """
    Main function to run evaluation
    """
    # Initialize evaluator
    evaluator = MarathiSTTEvaluator()

    # List of 15 Marathi audio files
    # Replace these paths with actual audio file paths
    audio_files = [
        "marathi_audio_01.wav",

    ]

    # Optional: Ground truth transcriptions
    ground_truths = None  # Add ground truth transcriptions if available

    # Run evaluation
    results = evaluator.evaluate_batch(audio_files, ground_truths)

    # Save results to text file
    evaluator.save_results(results, "marathi_stt_evaluation_results.txt")

    # Also save as JSON for programmatic access
    with open("marathi_stt_evaluation_results.json", 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(results)
    print("\nEvaluation complete!")


if __name__ == "__main__":
    main()

Loading model: ai4bharat/indic-conformer-600m-multilingual
Please check FRAME_DURATION_MS. The timestamps can be inaccurate
Please check FRAME_DURATION_MS. The timestamps can be inaccurate


Fetching 404 files:   0%|          | 0/404 [00:00<?, ?it/s]

Please check FRAME_DURATION_MS. The timestamps can be inaccurate


/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Model loaded on device: cuda

Processing audio 1/1: marathi_audio_01.wav


/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be r

Transcript: None

Results saved to marathi_stt_evaluation_results.txt
[{'audio_file': 'marathi_audio_01.wav', 'transcript': None, 'status': 'success'}]

Evaluation complete!


In [ ]:
from datasets import load_dataset

ds = load_dataset("rootflo/marathi-asr-data")

README.md:   0%|          | 0.00/509 [00:00<?, ?B/s]

data/train-00000-of-00005.parquet:   0%|          | 0.00/285M [00:00<?, ?B/s]

data/train-00001-of-00005.parquet:   0%|          | 0.00/584M [00:00<?, ?B/s]

data/train-00002-of-00005.parquet:   0%|          | 0.00/592M [00:00<?, ?B/s]

data/train-00003-of-00005.parquet:   0%|          | 0.00/442M [00:00<?, ?B/s]

data/train-00004-of-00005.parquet:   0%|          | 0.00/271M [00:00<?, ?B/s]

data/test-00000-of-00002.parquet:   0%|          | 0.00/502M [00:00<?, ?B/s]

data/test-00001-of-00002.parquet:   0%|          | 0.00/256M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7496 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2766 [00:00<?, ? examples/s]

In [ ]:
ds['train']

Dataset({
    features: ['audio', 'sentence', 'length'],
    num_rows: 7496
})

In [ ]:
from transformers import AutoModel
import torch, torchaudio

# Load the model
model = AutoModel.from_pretrained("ai4bharat/indic-conformer-600m-multilingual", trust_remote_code=True)

# Load an audio file


for audio , sentence, length in ds['train'][:15]:
    wav, sr = torchaudio.load(audio)
    wav = torch.mean(wav, dim=0, keepdim=True)

    target_sample_rate = 16000  # Expected sample rate
    if sr != target_sample_rate:
        resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=target_sample_rate)
        wav = resampler(wav)
    print("original senetence : "  ,sentence)
    # Perform ASR with CTC decoding
    transcription_ctc = model(wav, "mr", "ctc")
    print("CTC Transcription:", transcription_ctc)

    # Perform ASR with RNNT decoding
    transcription_rnnt = model(wav, "mr", "rnnt")
    print("RNNT Transcription:", transcription_rnnt)


KeyboardInterrupt: 

In [ ]:
!pip install transformers torchaudio onnx onnxruntime onnxruntime-gpu torchcodec


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 31.2 MB/s eta 0:00:00
